In [1]:
import os, re, html, json, random
from collections import Counter
import urllib.request
import tarfile

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

import nltk
from nltk.tokenize import TreebankWordTokenizer
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer
import string

In [2]:
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [3]:
# Seeds
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [4]:
# Device info
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch version: 2.8.0+cu126
CUDA available: True


In [5]:
# Set current directory as data root
data_root = "./dataset"  # Current directory
os.makedirs(data_root, exist_ok=True)

dataset_dir = os.path.join(data_root, "aclImdb")
tgz_path = os.path.join(data_root, "aclImdb_v1.tar.gz")
url = "http://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"

# Download and extract dataset if not present
if not os.path.exists(dataset_dir):
    if not os.path.exists(tgz_path):
        print("Downloading dataset to:", tgz_path)
        urllib.request.urlretrieve(url, tgz_path)
        print("Download finished.")
    
    print("Extracting to:", data_root)
    with tarfile.open(tgz_path, "r:gz") as tar:
        tar.extractall(path=data_root)
    print("Extraction finished.")
else:
    print("Dataset already present:", dataset_dir)

print("Using data path:", dataset_dir)
print("Path exists:", os.path.exists(dataset_dir))

Download finished.
Extracting to: ./dataset


C:\Users\Admin\AppData\Local\Temp\ipykernel_2388\2496988641.py:18: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=data_root)


Extraction finished.
Using data path: ./dataset\aclImdb
Path exists: True


In [6]:
# Check dirs
required_subdirs = ['train/pos', 'train/neg', 'test/pos', 'test/neg']
for sub in required_subdirs:
    p = os.path.join(dataset_dir, sub)
    print(f"{sub} exists:", os.path.exists(p))
    assert os.path.exists(p), f"Missing required subdir: {p}"

train/pos exists: True
train/neg exists: True
test/pos exists: True
test/neg exists: True


In [7]:
# Load raw train texts (pos/neg) and peek
def load_imdb_train_raw(data_path):
    pos_dir = os.path.join(data_path, 'train', 'pos')
    neg_dir = os.path.join(data_path, 'train', 'neg')
    texts, labels = [], []
    for filename in sorted(os.listdir(pos_dir)):
        if filename.endswith('.txt'):
            with open(os.path.join(pos_dir, filename), 'r', encoding='utf-8') as f:
                texts.append(f.read())
                labels.append(1)
    for filename in sorted(os.listdir(neg_dir)):
        if filename.endswith('.txt'):
            with open(os.path.join(neg_dir, filename), 'r', encoding='utf-8') as f:
                texts.append(f.read())
                labels.append(0)
    return texts, labels

raw_texts, raw_labels = load_imdb_train_raw(dataset_dir)
print(f"Loaded from train dir: {len(raw_texts)} samples")
print("Label distribution:", Counter(raw_labels))

# Peek raw samples
for i in range(2):
    print(f"\n[RAW {i}] {raw_texts[i][:300]}")

Loaded from train dir: 25000 samples
Label distribution: Counter({1: 12500, 0: 12500})

[RAW 0] Bromwell High is a cartoon comedy. It ran at the same time as some other programs about school life, such as "Teachers". My 35 years in the teaching profession lead me to believe that Bromwell High's satire is much closer to reality than is "Teachers". The scramble to survive financially, the insigh

[RAW 1] Homelessness (or Houselessness as George Carlin stated) has been an issue for years but never a plan to help those on the street that were once considered human who did everything from going to school, work, or vote for the matter. Most people think of the homeless as just a lost cause while worryin


In [8]:
# Define cleaner, tokenizer, negation handler and test on few samples
def clean_text(text):
    text = re.sub(r'<[^>]+>', ' ', text)      # remove HTML tags
    text = html.unescape(text)                 # decode HTML entities
    text = text.lower()
    text = re.sub(r"([!?.])", r" \1 ", text)  # separate punctuation
    text = re.sub(r"[^a-z0-9'!?.\s]", " ", text)  # keep letters, digits, apostrophes, ! ? .
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize(text):
    # NLTK
    tokenizer = TreebankWordTokenizer()
    return tokenizer.tokenize(text)

def handle_negation(tokens):
    out, negate = [], False
    neg_triggers = {"not", "n't", "no", "never"}
    end_punct = {"!", "?", "."}
    for tok in tokens:
        if tok in neg_triggers:
            out.append(tok)
            negate = True
            continue
        if tok in end_punct:
            out.append(tok)
            negate = False
            continue
        if negate and tok.isalpha():
            out.append("NOT_" + tok)
        else:
            out.append(tok)
    return out

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()

def extra_preprocess(tokens):
    tokens = [t for t in tokens if t not in string.punctuation]
    # Remove stopwords
    tokens = [t for t in tokens if t.lower() not in stop_words]
    # Drop standalone contractions or meaningless short tokens
    tokens = [t for t in tokens if t.lower() not in {"'s", "'t", "'re", "'ve", "'m", "'d"}]
    # Lemmatization
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return tokens

# Try on 2 samples
for i in range(2):
    print(f"\n[Cleaned {i}] {clean_text(raw_texts[i])[:300]}")
    toks = tokenize(clean_text(raw_texts[i]))
    print(f"[Tokens {i}] {toks[:40]}")
    toks_neg = handle_negation(toks)
    print(f"[Tokens+Neg {i}] {toks_neg[:40]}")
    toks_extra = extra_preprocess(toks_neg)
    print(f"[Tokens+Neg+Extra {toks_extra[:40]}")


[Cleaned 0] bromwell high is a cartoon comedy . it ran at the same time as some other programs about school life such as teachers . my 35 years in the teaching profession lead me to believe that bromwell high's satire is much closer to reality than is teachers . the scramble to survive financially the insightfu
[Tokens 0] ['bromwell', 'high', 'is', 'a', 'cartoon', 'comedy', '.', 'it', 'ran', 'at', 'the', 'same', 'time', 'as', 'some', 'other', 'programs', 'about', 'school', 'life', 'such', 'as', 'teachers', '.', 'my', '35', 'years', 'in', 'the', 'teaching', 'profession', 'lead', 'me', 'to', 'believe', 'that', 'bromwell', 'high', "'s", 'satire']
[Tokens+Neg 0] ['bromwell', 'high', 'is', 'a', 'cartoon', 'comedy', '.', 'it', 'ran', 'at', 'the', 'same', 'time', 'as', 'some', 'other', 'programs', 'about', 'school', 'life', 'such', 'as', 'teachers', '.', 'my', '35', 'years', 'in', 'the', 'teaching', 'profession', 'lead', 'me', 'to', 'believe', 'that', 'bromwell', 'high', "'s", 'satire']
[Tok

In [9]:
# Clean + tokenize entire corpus, inspect length distribution
cleaned_texts = [clean_text(t) for t in raw_texts]
tokens_list = [tokenize(t) for t in cleaned_texts]
tokens_list = [handle_negation(toks) for toks in tokens_list]  # optional but recommended
tokens_list = [extra_preprocess(toks) for toks in tokens_list]

lengths = np.array([len(t) for t in tokens_list])
print("Total tokens:", sum(lengths))
print("Num docs:", len(tokens_list))
print("Length stats -> mean:", lengths.mean(), "median:", np.median(lengths))
for p in [90, 95, 99]:
    print(f"P{p}:", int(np.percentile(lengths, p)))

Total tokens: 3491363
Num docs: 25000
Length stats -> mean: 139.65452 median: 104.0
P90: 277
P95: 364
P99: 546


In [10]:
# Split (train/val/test) from the training dir with stratify
split_ratios = (0.7, 0.15, 0.15)
train_ratio, val_ratio, test_ratio = split_ratios
assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6

# First split out test
all_indices = np.arange(len(tokens_list))
temp_idx, test_idx, temp_labels, test_labels = train_test_split(
    all_indices, raw_labels, test_size=test_ratio, random_state=SEED, stratify=raw_labels
)
# From temp, split val
val_size = val_ratio / (train_ratio + val_ratio)
train_idx, val_idx, train_labels, val_labels = train_test_split(
    temp_idx, temp_labels, test_size=val_size, random_state=SEED, stratify=temp_labels
)

print(f"Split sizes -> train: {len(train_idx)}, val: {len(val_idx)}, test: {len(test_idx)}")
print("Train label dist:", Counter(train_labels))
print("Val label dist:", Counter(val_labels))
print("Test label dist:", Counter(test_labels))

Split sizes -> train: 17499, val: 3751, test: 3750
Train label dist: Counter({1: 8750, 0: 8749})
Val label dist: Counter({0: 1876, 1: 1875})
Test label dist: Counter({0: 1875, 1: 1875})


In [11]:
# Build vocabulary from train tokens only
def build_vocab_from_tokens(tokens_lists, min_freq=5, max_size=None):
    counter = Counter()
    for toks in tokens_lists:
        counter.update(toks)
    # special tokens
    vocab = {'<PAD>': 0, '<UNK>': 1}
    items = sorted(counter.items(), key=lambda x: (-x[1], x[0]))
    added = 0
    for w, c in items:
        if c < min_freq:
            continue
        if max_size is not None and added >= max_size:
            break
        vocab[w] = len(vocab)
        added += 1
    return vocab, items

train_tokens = [tokens_list[i] for i in train_idx]
vocab, sorted_items = build_vocab_from_tokens(train_tokens, min_freq=5, max_size=50000)
print("Vocabulary size (incl PAD/UNK):", len(vocab))
print("Top 20 frequent words:", sorted_items[:20])

Vocabulary size (incl PAD/UNK): 28083
Top 20 frequent words: [('NOT_the', 31932), ('movie', 30605), ('film', 27901), ("n't", 23526), ('NOT_to', 18310), ('NOT_a', 16632), ('one', 16026), ('NOT_of', 14590), ('NOT_and', 14222), ('NOT_it', 12332), ('like', 11956), ('NOT_in', 9962), ('time', 9267), ('good', 8594), ('NOT_that', 8593), ('character', 8409), ('would', 7905), ('story', 7852), ('NOT_this', 7764), ('NOT_is', 7406)]


In [12]:
# UNK coverage check
def compute_unk_rate_tokens(tokens_lists, vocab):
    total = 0
    unk = 0
    for toks in tokens_lists:
        total += len(toks)
        unk += sum(1 for w in toks if w not in vocab)
    rate = (unk / total) if total else 0.0
    return rate, unk, total

val_tokens = [tokens_list[i] for i in val_idx]
test_tokens = [tokens_list[i] for i in test_idx]

for name, tl in [('train', train_tokens), ('val', val_tokens), ('test', test_tokens)]:
    rate, unk, total = compute_unk_rate_tokens(tl, vocab)
    print(f"{name} UNK rate: {rate:.4f} ({unk}/{total})")

train UNK rate: 0.0379 (92602/2445782)
val UNK rate: 0.0458 (23623/516191)
test UNK rate: 0.0463 (24535/529390)


In [13]:
# Choose max_length by percentile and define truncation
train_lengths = np.array([len(t) for t in train_tokens])
suggested_max_len = int(np.percentile(train_lengths, 95))
print("Suggested max_length (P95 on train):", suggested_max_len)

def head_tail_truncate(seq, max_len, head_ratio=0.5):
    if len(seq) <= max_len:
        return seq
    head_len = int(max_len * head_ratio)
    tail_len = max_len - head_len
    return seq[:head_len] + seq[-tail_len:]

Suggested max_length (P95 on train): 364


In [14]:
# Encode tokens to ids with truncation
PAD_ID = 0
UNK_ID = 1
MAX_LEN = suggested_max_len

def encode_tokens(tokens, vocab, max_len=None):
    if max_len is not None:
        tokens = head_tail_truncate(tokens, max_len)
    return [vocab.get(w, UNK_ID) for w in tokens]

train_ids = [encode_tokens(tokens_list[i], vocab, MAX_LEN) for i in train_idx]
val_ids   = [encode_tokens(tokens_list[i], vocab, MAX_LEN) for i in val_idx]
test_ids  = [encode_tokens(tokens_list[i], vocab, MAX_LEN) for i in test_idx]

# Peek encoded samples
print("Sample train ids (len, first 20):", len(train_ids[0]), train_ids[0][:20])

Sample train ids (len, first 20): 58 [1321, 218, 4, 3846, 236, 15, 1464, 11628, 3827, 1474, 374, 6846, 1199, 10615, 17, 473, 71, 5270, 4, 419]


In [15]:
# Dataset (var-length) + dynamic padding collate_fn + DataLoaders
class IMDBVarLenDataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences = sequences
        self.labels = labels
    def __len__(self):
        return len(self.sequences)
    def __getitem__(self, idx):
        return torch.tensor(self.sequences[idx], dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.long)

def collate_fn(batch, pad_id=PAD_ID):
    seqs, labels = zip(*batch)  # tuple of tensors (var length)
    lengths = torch.tensor([len(s) for s in seqs], dtype=torch.long)
    padded = torch.nn.utils.rnn.pad_sequence(seqs, batch_first=True, padding_value=pad_id)
    mask = (padded != pad_id).long()
    labels = torch.stack(labels) 
    return padded, labels, lengths, mask

batch_size = 32
train_dataset = IMDBVarLenDataset(train_ids, train_labels)
val_dataset   = IMDBVarLenDataset(val_ids,   val_labels)
test_dataset  = IMDBVarLenDataset(test_ids,  test_labels)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                          collate_fn=lambda b: collate_fn(b, pad_id=PAD_ID), num_workers=0)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,
                          collate_fn=lambda b: collate_fn(b, pad_id=PAD_ID), num_workers=0)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False,
                          collate_fn=lambda b: collate_fn(b, pad_id=PAD_ID), num_workers=0)

# Sanity check one batch
for i, (data, target, lengths, mask) in enumerate(train_loader):
    print(f"[Sanity Check] batch {i+1}: data{tuple(data.shape)}, target{tuple(target.shape)}, lengths{tuple(lengths.shape)}, mask{tuple(mask.shape)}")
    print("First sample length:", lengths[0].item())
    print("First sample ids (first 20):", data[0][:20].tolist())
    print("First sample mask (first 20):", mask[0][:20].tolist())
    print("First sample label:", target[0].item())
    break

[Sanity Check] batch 1: data(32, 364), target(32,), lengths(32,), mask(32, 364)
First sample length: 90
First sample ids (first 20): [1, 54, 28, 242, 959, 38, 153, 1596, 200, 132, 33, 28, 889, 37, 102, 42, 182, 33, 33, 23]
First sample mask (first 20): [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
First sample label: 0


In [16]:
# Persist artifacts for reuse
data_root = "./dataset" 
artifacts_dir = os.path.join(data_root, "imdb_artifacts")
os.makedirs(artifacts_dir, exist_ok=True)

vocab_path = os.path.join(artifacts_dir, "vocab.json")
with open(vocab_path, "w", encoding="utf-8") as f:
    json.dump(vocab, f, ensure_ascii=False)

print("Saved vocab to:", vocab_path)

Saved vocab to: ./dataset\imdb_artifacts\vocab.json


In [17]:
# Save training/validation/test sets
torch.save((train_ids, train_labels), os.path.join(artifacts_dir, "train_dataset.pt"))
torch.save((val_ids, val_labels), os.path.join(artifacts_dir, "val_dataset.pt"))
torch.save((test_ids, test_labels), os.path.join(artifacts_dir, "test_dataset.pt"))

In [18]:
print("Saved train to:", os.path.join(artifacts_dir, "train_dataset.pt"))
print("Saved val to:", os.path.join(artifacts_dir, "val_dataset.pt"))
print("Saved test to:", os.path.join(artifacts_dir, "test_dataset.pt"))

Saved train to: ./dataset\imdb_artifacts\train_dataset.pt
Saved val to: ./dataset\imdb_artifacts\val_dataset.pt
Saved test to: ./dataset\imdb_artifacts\test_dataset.pt


In [19]:
# Read data
train_path = os.path.join(artifacts_dir, "train_dataset.pt")
val_path = os.path.join(artifacts_dir, "val_dataset.pt")
test_path = os.path.join(artifacts_dir, "test_dataset.pt")

train_ids_new, train_labels_new = torch.load(train_path)
val_ids_new, val_labels_new = torch.load(val_path)
test_ids_new, test_labels_new = torch.load(test_path)

In [20]:
# Rebuild Dataset
train_dataset_new = IMDBVarLenDataset(train_ids_new, train_labels_new)
val_dataset_new = IMDBVarLenDataset(val_ids_new, val_labels_new)
test_dataset_new = IMDBVarLenDataset(test_ids_new, test_labels_new)

In [21]:
# Rebuild DataLoader
train_loader_new = DataLoader(train_dataset_new, batch_size=batch_size, shuffle=True,
collate_fn=lambda b: collate_fn(b, pad_id=PAD_ID), num_workers=0)
val_loader_new = DataLoader(val_dataset_new, batch_size=batch_size, shuffle=False,
collate_fn=lambda b: collate_fn(b, pad_id=PAD_ID), num_workers=0)
test_loader_new = DataLoader(test_dataset_new, batch_size=batch_size, shuffle=False,
collate_fn=lambda b: collate_fn(b, pad_id=PAD_ID), num_workers=0)

In [22]:
vocab_path = os.path.join(artifacts_dir, "vocab.json")

# Read JSON
with open(vocab_path, "r", encoding="utf-8") as f:
    vocab = json.load(f)

id2word = {int(idx): word for word, idx in vocab.items()}

In [23]:
for batch_data, batch_labels, batch_lengths, batch_mask in train_loader_new:
    i = random.randint(0, batch_data.size(0) - 1)
    sample_ids = batch_data[i][:20].tolist()  # 前20个 token id
    print("Sample data (ids):", sample_ids)
    print("Sample label:", batch_labels[i].item())
    print("Sequence length:", batch_lengths[i].item())
    print("Mask (first 20):", batch_mask[i][:20].tolist())
    sample_words = [id2word.get(idx, "<UNK>") for idx in sample_ids]
    print("Sample data (words):", sample_words)
    break

Sample data (ids): [644, 1112, 3, 2282, 8, 2992, 953, 47, 269, 7, 861, 9456, 1213, 22, 30, 304, 3112, 174, 347, 7]
Sample label: 1
Sequence length: 88
Mask (first 20): [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Sample data (words): ['happened', 'catch', 'movie', 'cable', 'one', 'afternoon', 'admit', 'never', 'NOT_been', 'NOT_a', 'NOT_big', 'NOT_baseball', 'NOT_fan', 'NOT_but', 'NOT_i', 'NOT_can', 'NOT_sometimes', 'NOT_get', 'NOT_into', 'NOT_a']
